# 0.1 - Data preparation

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from itertools import combinations
from matplotlib.colors import LinearSegmentedColormap

from evaluation import DIVERGING, SEQUENTIAL_BLUE, SERIES_COLOR

## Constants

In [ ]:
ROOT = '../../'
RANDOM_SEED = 42

# Plausibility bounds
MIN_CONSTRUCTION_YEAR = 900
MAX_CONSTRUCTION_YEAR = 2025  # year of the radon database extract
FLOOR_RANGE = (-5, 50)        # floor of the measured room
MAX_FLOORS = 50               # number of floors of the building

# No-data values encoded as numbers in the source rasters/layers
TMUD_NO_DATA = -999999999
KARST_NO_DATA = -9999
RADON_MAP_NO_DATA = -1.7e+308

## Load dataset

In [ ]:
df = pd.read_csv(f'{ROOT}data/final/dataset-v1.0.csv')

df.head()

## Remove duplicates from spatial joins

In [ ]:
# Spatial joins (karst, forest, ...) duplicate records that intersect several polygons; keep one row per measurement
df = df.drop_duplicates(subset='orig_idx')

## Keep valid measurements

Duration of at least 90 days and a measured radon concentration (the target).

In [ ]:
df = df[df['Rdn - duration_days'] >= 90]
df = df[df['Rdn - Concentration de radon (Bq/m3)_BQ_M3'].notna()]

## Drop columns that are unnecessary for modelling

In [ ]:
# List columns
df.columns.tolist()

In [ ]:
columns_to_keep = [
# 'orig_idx',
# 'Rdn - Canton',
# 'Rdn - ID maison',
# 'EGID',
 'Rdn - Commune',
 'Rdn - Catégorie de bâtiment',
 'Rdn - Année de construction',
 'Rdn - Nombre étages',
 'Rdn - Fondation',
 'Rdn - Structure de fondation',
 'Rdn - Nombre étage sous-terrain',
 'Rdn - Terrain en pente',
 'Rdn - Ventilation contrôlée',
#  'Rdn - LV95_ORDINATE',
#  'Rdn - LV95_ABSZISSE',
# 'Rdn - LV03_ORDINATE',
# 'Rdn - LV03_ABSZISSE',
# 'Rdn - Protocole de mesure',
#  'Rdn - Type de mesure',
#  'Rdn - Type instrument de mesure',
# 'Rdn - ID local',
 'Rdn - Type de local',
# 'Rdn - Séjour de personnes',
# 'Rdn - Heures de séjour par an',
 'Rdn - Etage',
# 'Rdn - Exposition de radon (kBqh/m3)',
# 'Rdn - Incertitude (kBqh/m3)',
 'Rdn - Concentration de radon (Bq/m3)_BQ_M3',
# 'Rdn - duration_days',  # describes the measurement, unknown when screening a building
#  'Rdn - Début_year',
#  'Rdn - Début_month',
#  'Rdn - Début_day',
#  'Rdn - Fin_year',
#  'Rdn - Fin_month',
#  'Rdn - Fin_day',
#  'RegBL - Canton',
#  'RegBL - Nom de la commune',
#  'RegBL - Numéro d’immeuble',
#  'RegBL - Numéro officiel du bâtiment',
#  'RegBL - Désignation du bâtiment',
 'Coordonnée E',
 'Coordonnée N',
#  'RegBL - Provenance des coordonnées',
#  'RegBL - Statut du bâtiment',
# 'RegBL - Catégorie de bâtiment',
 'RegBL - Classe de bâtiment',
# 'RegBL - Année de construction du bâtiment',
# 'RegBL - Mois de construction du bâtiment',
 'RegBL - Période de construction',
# 'RegBL - Année de démolition du bâtiment',
 'RegBL - Surface du bâtiment',
 'RegBL - Volume du bâtiment',
# 'RegBL - Nombre de niveaux',
 'RegBL - Nombre de logements',
 'RegBL - Nombre de pièces d’hab. indép.',
 'RegBL - Abri de protection civile',
 'RegBL - Surface de référence énergétique',
 'RegBL - Générateur de chaleur pour le chauffage 1',
 'RegBL - Source d’énergie / de chaleur pour le chauffage 1',
 'RegBL - Générateur de chaleur pour le chauffage 2',
 'RegBL - Source d’énergie / de chaleur pour le chauffage 2',
 "RegBL - Générateur de chaleur pour l'eau chaude 1",
 "RegBL - Source d’énergie / de chaleur pour l'eau chaude 1",
 "RegBL - Générateur de chaleur pour l'eau chaude 2",
 "RegBL - Source d’énergie / de chaleur pour l'eau chaude 2",
 'Ura - Concentration uranium',
 'Clim - Durée ensoleillement',
 'Clim - Précipitation',
 'Clim - Température',
 'UncSoil - Classe de sol',
 'Relief - Altitude (m)',
# 'Miner - Certification de minergie',  # certificate number (identifier)
 'Miner - Standard de minergie',
 'Rdn Map - Probabilité dépassement valeur réf radon',
 'Rdn Map - Index de confiance dépassement valeur réf',
#  'ArealStat - RELI',
#  'ArealStat - E_COORD',
#  'ArealStat - N_COORD',
#  'ArealStat - GMDE_ID',
#  'ArealStat - GMDE_HISTID',
#  'ArealStat - FJ85',
#  'ArealStat - FJ97',
#  'ArealStat - FJ09',
#  'ArealStat - FJ18',
#  'ArealStat - FJ25',
#  'ArealStat - METHOD25',
#  'ArealStat - REVISION25',
#  'ArealStat - AS85_72',
#  'ArealStat - AS97_72',
#  'ArealStat - AS09_72',
#  'ArealStat - AS18_72',
#  'ArealStat - AS25_72',
#  'ArealStat - AS85_27',
#  'ArealStat - AS97_27',
#  'ArealStat - AS09_27',
#  'ArealStat - AS18_27',
#  'ArealStat - AS25_27',
#  'ArealStat - AS85_17',
#  'ArealStat - AS97_17',
#  'ArealStat - AS09_17',
#  'ArealStat - AS18_17',
#  'ArealStat - AS25_17',
#  'ArealStat - AS85_4',
#  'ArealStat - AS97_4',
#  'ArealStat - AS09_4',
#  'ArealStat - AS18_4',
#  'ArealStat - AS25_4',
#  'ArealStat - LC85_27',
#  'ArealStat - LC97_27',
#  'ArealStat - LC09_27',
#  'ArealStat - LC18_27',
#  'ArealStat - LC25_27',
#  'ArealStat - LC85_6',
#  'ArealStat - LC97_6',
#  'ArealStat - LC09_6',
#  'ArealStat - LC18_6',
#  'ArealStat - LC25_6',
#  'ArealStat - LU85_46',
#  'ArealStat - LU97_46',
#  'ArealStat - LU09_46',
#  'ArealStat - LU18_46',
#  'ArealStat - LU25_46',
#  'ArealStat - LU85_10',
#  'ArealStat - LU97_10',
#  'ArealStat - LU09_10',
#  'ArealStat - LU18_10',
#  'ArealStat - LU25_10',
#  'ArealStat - LU85_4',
#  'ArealStat - LU97_4',
#  'ArealStat - LU09_4',
#  'ArealStat - LU18_4',
#  'ArealStat - LU25_4',
#  'distance_to_stats',
# 'HydroWM -  GWKID',
 'HydroWM -  GWKName',
 'HydroWM -  Flussgebiet',
 'HydroWM -  Grundwasserleitertyp',
 'HydroWM -  Naturraum',
 'HydroWM -  Grundwasserregime',
 'HydroWM -  Area',
 'HydroWM -  SHAPE_Length',
# 'HydroWM -  SHAPE_Area',
# 'HydroWM -  Farbe',  # map colour code
 'Seism - ZONE',
 'Geol - LITHO',
 'Geol - PERIODE',
 'Geol - EPOQUE',
 'Geol - ETAGE',
 'Geol - LEG_TEC_1',
 'Geol - LEG_TEC_2',
 'Geol - LEG_TEC_3',
 'Geol - AQUIFERE',
 'Geol - HYDROGEOL',
# 'Geol - ID_LITH',
 'Geol - LITH_PET',
# 'Geol - ID_GENESE',
 'Geol - GENESE',
# 'Geol - ID_TYP_ROC',
 'Geol - TYPE_ROCHE',
 'Geol - COUVERTURE',
 'Geol - Distance to fault (m)',
# 'geometry',
# 'Karst - BA_ID',
 'Karst - BA_Name',
 'Karst - BA_Alt_min',
 'Karst - BA_Alt_moy',
 'Karst - BA_Alt_max',
 'Karst - BA_UStruct',
# 'Karst - BA_DD',  # empty
# 'Karst - BA_Canton',
# 'Karst - BA_Rem',  # free-text remarks, almost empty
 'Karst - BA_Aqui',
 'Karst - SHAPE_Length',
 'Karst - SHAPE_Area',
 'Karst - BA_Surf',
# 'Karst - UB_ID',
# 'Karst - UB_Name',
 'Karst - UB_Type',
 'Karst - UB_Flux',
 'Karst - UB_Alt_min',
 'Karst - UB_Alt_moy',
 'Karst - UB_Alt_max',
 'Shape_Length_left',
 'Shape_Area_left',
# 'Karst - FID_NK',  # identifier
# 'Karst - NK_ID',
# 'Karst - NK_Hdyn',  # single value
 'Karst - NK_Type',
# 'Karst - NK_Level',  # single value
# 'Karst - MNK_ID',
 'Shape_Length_right',
 'Shape_Area_right',
# 'Forest - Region_de',
 'Forest - Region_fr',
# 'Forest - Region_it',
# 'Forest - Region_en',
# 'Forest - Code',
# 'Forest - Subcode',
# 'Forest - Code_Bu',
# 'Forest - Code_Fi',
 'Forest - Shape_Leng',
 'Forest - Shape_Area',
# 'index_right',
#  'TBB - CH_CODE_HN',
#  'TBB - CH_BEZ_D',
#  'TBB - CH_BEZ_F',
#  'TBB - BFS_NO',
#  'TBB - NAME',
#  'TBB - KT_NO',
#  'TBB - KT_KZ',
#  'TBB - FLAECHE'
]

# Kept aside, not predictors: building IDs to split the data by building, and whether the room was inhabited
# (long stays, as for the reference level) to compare with the radon map, which refers to inhabited rooms
egid = df['EGID']
inhabited = df['Rdn - Séjour de personnes'] == 'YES_LONG'

df = df[columns_to_keep]

In [ ]:
df.columns.tolist()

## Analyze correlations

In [ ]:
def analyze_correlations(df):
    # Identify column types
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    
    # Try converting columns that look numeric but are stored as strings
    for col in cat_cols[:]:
        try:
            df[col] = pd.to_numeric(df[col])
            num_cols.append(col)
            cat_cols.remove(col)
        except ValueError:
            pass
    
    # Numerical correlations (Pearson)
    num_corr = None
    if num_cols:
        num_corr = df[num_cols].corr(method='pearson')

    # Categorical correlations (Cramer's V)
    def cramers_v(x, y):
        confusion = pd.crosstab(x, y)
        from scipy.stats import chi2_contingency
        chi2 = chi2_contingency(confusion)[0]
        n = confusion.sum().sum()
        phi2 = chi2 / n
        r, k = confusion.shape
        phi2corr = max(0, phi2 - ((k-1)*(r-1)) / (n-1))
        rcorr = r - ((r-1)**2) / (n-1)
        kcorr = k - ((k-1)**2) / (n-1)
        return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))
    
    cat_corr = None
    if len(cat_cols) > 1:
        cat_encoded = df[cat_cols].apply(lambda x: x.astype(str))
        matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
        for col1, col2 in combinations(cat_cols, 2):
            v = cramers_v(cat_encoded[col1], cat_encoded[col2])
            matrix.loc[col1, col2] = v
            matrix.loc[col2, col1] = v
        np.fill_diagonal(matrix.values, 1)
        cat_corr = matrix
    
    return num_corr, cat_corr

num_corr, cat_corr = analyze_correlations(df)

In [ ]:
# Plot numerical correlations
if num_corr is not None:
    plt.figure(figsize=(12, 10))
    sns.heatmap(num_corr, cmap=LinearSegmentedColormap.from_list('diverging', DIVERGING), vmin=-1, vmax=1, center=0, square=True)
    plt.title('Numerical Feature Correlations')
    plt.show()

# Plot categorical correlations
if cat_corr is not None:
    plt.figure(figsize=(12, 10))
    sns.heatmap(cat_corr.astype(float), cmap=LinearSegmentedColormap.from_list('blue', SEQUENTIAL_BLUE), vmin=0, vmax=1, square=True)
    plt.title("Categorical Feature Correlations (Cramer's V)")
    plt.show()

## Handle missing values

Categorical gaps are filled with 'Unknown' here. Numerical gaps stay missing: they are imputed by the 0.2 pipeline, fitted on the training data only.

In [ ]:
# Drop rows with an implausible construction year (data-entry errors such as 9999)
df = df[df['Rdn - Année de construction'].between(MIN_CONSTRUCTION_YEAR, MAX_CONSTRUCTION_YEAR, inclusive='right')]

In [ ]:
def set_implausible_to_nan(data):
    """Replace implausible values and numeric no-data codes with NaN so they are imputed like other missing values."""
    data = data.copy()
    data.loc[~data['Rdn - Etage'].between(*FLOOR_RANGE), 'Rdn - Etage'] = np.nan
    data.loc[data['Rdn - Nombre étages'] > MAX_FLOORS, 'Rdn - Nombre étages'] = np.nan
    data['UncSoil - Classe de sol'] = data['UncSoil - Classe de sol'].replace(TMUD_NO_DATA, np.nan)
    data['Karst - BA_UStruct'] = data['Karst - BA_UStruct'].replace(KARST_NO_DATA, np.nan)
    for col in ['Rdn Map - Probabilité dépassement valeur réf radon', 'Rdn Map - Index de confiance dépassement valeur réf']:
        data[col] = data[col].replace(RADON_MAP_NO_DATA, np.nan)
    return data

df = set_implausible_to_nan(df)

In [ ]:
# Check for missing values
missing_percentage = df.isnull().mean() * 100

# Plot missing values percentage
plt.figure(figsize=(12, 6))
missing_percentage[missing_percentage > 0].sort_values().plot(kind='bar', color=SERIES_COLOR)
plt.title('Missing Values Percentage by Feature')
plt.ylabel('Percentage (%)')
plt.xlabel('Features')
plt.show()

In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()

categorical_columns, numerical_columns

In [ ]:
for col in categorical_columns:
    df[col] = df[col].fillna('Unknown').replace({'Inconnu': 'Unknown', '?': 'Unknown'})

In [ ]:
# Numerical inf values become missing, like other numerical gaps
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
df.shape

## Split data into train and test sets

In [ ]:
y = df['Rdn - Concentration de radon (Bq/m3)_BQ_M3']
X = df.drop(columns=[
    'Rdn - Concentration de radon (Bq/m3)_BQ_M3'])

In [ ]:
X.shape, y.shape

In [ ]:
# Split by building so that measurements from the same building never end up in both train and test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=egid.loc[X.index]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

assert not set(egid.loc[X_train.index]) & set(egid.loc[X_test.index]), 'Buildings shared between train and test'

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## Save data

In [ ]:
X_train.to_csv(f"{ROOT}data/final/dataset-v0.2-x-train.csv")
X_test.to_csv(f"{ROOT}data/final/dataset-v0.2-x-test.csv")

y_train.to_csv(f"{ROOT}data/final/dataset-v0.2-y-train.csv")
y_test.to_csv(f"{ROOT}data/final/dataset-v0.2-y-test.csv")

# Building IDs: the validation set is split off by building, and confidence intervals resample buildings
egid.loc[X_train.index].to_csv(f"{ROOT}data/final/dataset-v0.2-egid-train.csv")
egid.loc[X_test.index].to_csv(f"{ROOT}data/final/dataset-v0.2-egid-test.csv")

# Inhabited-room flag, used only to evaluate against the radon map
inhabited.loc[X_train.index].rename('inhabited').to_csv(f"{ROOT}data/final/dataset-v0.2-inhabited-train.csv")
inhabited.loc[X_test.index].rename('inhabited').to_csv(f"{ROOT}data/final/dataset-v0.2-inhabited-test.csv")